# Test predict-then-optimise baselines

This notebook is a small end-to-end smoke test for `models/src/pto_baselines.py`: load the macro and charge-off data, fit the linear and MLP PTO models on a chronological training sample, and evaluate them out of sample. It is intentionally simple before moving to an expanding-window setup.

In [ ]:
from pathlib import Path
import importlib.util
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler

# Make the notebook robust whether it is launched from the repo root or models/notebooks.
ROOT = Path.cwd().resolve()
while ROOT != ROOT.parent and not (ROOT / "models" / "src" / "pto_baselines.py").exists():
    ROOT = ROOT.parent

if not (ROOT / "models" / "src" / "pto_baselines.py").exists():
    raise FileNotFoundError("Could not locate repo root containing models/src/pto_baselines.py")

missing = [pkg for pkg in ["torch", "scipy"] if importlib.util.find_spec(pkg) is None]
if missing:
    raise ImportError(
        f"Missing required package(s): {missing}. Install the project requirements with: "
        f"pip install -r {ROOT / 'requirements.txt'}"
    )

sys.path.insert(0, str(ROOT / "models" / "src"))

from pto_baselines import (
    CapitalConfig,
    ModelConfig,
    build_prediction_frame,
    chronological_split,
    evaluate_predictions,
    load_and_align_data,
    predict_p,
    train_pto_model,
)

pd.set_option("display.max_columns", 80)
pd.set_option("display.float_format", "{:.6f}".format)

ROOT

## Load and align data

`macro_panel_quarterly.csv` is dated at quarter end, while `fred_loan_return_aligned.csv` is dated at quarter start. The temporary macro copy below converts macro dates to quarter start before using the baseline `load_and_align_data` function.

In [ ]:
macro_csv = ROOT / "data" / "processed" / "macro_panel_quarterly.csv"
chargeoff_csv = ROOT / "data" / "raw" / "fred_loan_return" / "fred_loan_return_aligned.csv"
scratch_dir = ROOT / "models" / "notebooks" / ".scratch"
scratch_dir.mkdir(parents=True, exist_ok=True)

macro = pd.read_csv(macro_csv)
macro["DATE"] = pd.to_datetime(macro["DATE"]).dt.to_period("Q").dt.start_time
macro_aligned_csv = scratch_dir / "macro_panel_quarter_start.csv"
macro.to_csv(macro_aligned_csv, index=False)

df = load_and_align_data(
    macro_csv=macro_aligned_csv,
    chargeoff_csv=chargeoff_csv,
    date_col="DATE",
    target_col="charge_off_rate",
    feature_cols=None,
    horizon=1,
    target_is_percent=True,
)

feature_cols = df.attrs["feature_cols"]
print(f"Observations after alignment and one-step target: {len(df)}")
print(f"Date range: {df['DATE'].min().date()} to {df['DATE'].max().date()}")
print(f"Features ({len(feature_cols)}): {feature_cols}")

display(df.head())
display(df.tail())

assert len(df) > 40, "Expected enough quarterly observations for a train/test smoke test."
assert df[feature_cols + ["target_chargeoff_next"]].notna().all().all()

## Chronological train/test split

The split is deliberately chronological to mimic a real out-of-sample evaluation.

In [ ]:
train_frac = 0.75
train_df, test_df = chronological_split(df, train_frac=train_frac)

scaler = StandardScaler()
x_train = scaler.fit_transform(train_df[feature_cols].to_numpy(dtype=float))
x_test = scaler.transform(test_df[feature_cols].to_numpy(dtype=float))
y_train = train_df["target_chargeoff_next"].to_numpy(dtype=float)
y_test = test_df["target_chargeoff_next"].to_numpy(dtype=float)

print(f"Train rows: {len(train_df)} | {train_df['DATE'].min().date()} to {train_df['DATE'].max().date()}")
print(f"Test rows:  {len(test_df)} | {test_df['DATE'].min().date()} to {test_df['DATE'].max().date()}")
print(f"x_train: {x_train.shape}, x_test: {x_test.shape}")

assert x_train.shape[1] == len(feature_cols)
assert len(y_train) == len(train_df)
assert len(y_test) == len(test_df)

## Train linear and MLP PTO models

The epoch count is modest so this remains quick to rerun while checking whether the pipeline works.

In [ ]:
cap_cfg = CapitalConfig()
model_cfg = ModelConfig(
    hidden_dim=16,
    lr=1e-3,
    weight_decay=1e-4,
    batch_size=16,
    epochs=500,
    patience=75,
    val_frac=0.2,
    seed=42,
)

metrics = {}
prediction_frames = []
models = {}

for model_type in ["linear", "mlp"]:
    model = train_pto_model(
        x_train=x_train,
        y_train_chargeoff=y_train,
        model_type=model_type,
        model_cfg=model_cfg,
        cap_cfg=cap_cfg,
    )
    p_test = predict_p(model, x_test)
    metrics[model_type] = evaluate_predictions(y_test, p_test, cap_cfg)
    prediction_frames.append(
        build_prediction_frame(
            dates=test_df["DATE"],
            y_true=y_test,
            p_hat=p_test,
            model_name=model_type,
            cap_cfg=cap_cfg,
        )
    )
    models[model_type] = model

metrics_df = pd.DataFrame(metrics).T
display(metrics_df)

predictions = pd.concat(prediction_frames, ignore_index=True)
display(predictions.head())

assert predictions["p_hat_stress"].between(0, 1).all()
assert predictions["alpha_hat"].between(0, 1).all()
assert np.isfinite(metrics_df.to_numpy()).all()

## Out-of-sample diagnostics

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(12, 10), sharex=True)

for model_name, group in predictions.groupby("model"):
    axes[0].plot(group["date"], group["mu_hat_chargeoff"], label=f"{model_name} forecast")
    axes[1].plot(group["date"], group["alpha_hat"], label=f"{model_name} alpha")
    axes[2].plot(group["date"], group["regret"], label=f"{model_name} regret")

axes[0].plot(test_df["DATE"], y_test, color="black", linewidth=2, label="realised next charge-off")
axes[1].plot(predictions.loc[predictions["model"] == "linear", "date"], predictions.loc[predictions["model"] == "linear", "alpha_oracle"], color="black", linewidth=2, label="oracle alpha")

axes[0].set_ylabel("Charge-off rate")
axes[1].set_ylabel("Alpha")
axes[2].set_ylabel("Regret")
axes[2].set_xlabel("Date")

for ax in axes:
    ax.grid(True, alpha=0.25)
    ax.legend(loc="best")

fig.suptitle("Out-of-sample PTO smoke test", y=0.995)
plt.tight_layout()
plt.show()

## Save optional outputs

This gives you a small CSV to inspect or compare against future expanding-window results.

In [ ]:
output_csv = scratch_dir / "pto_baseline_smoke_test_predictions.csv"
predictions.to_csv(output_csv, index=False)
print(f"Saved predictions to {output_csv.relative_to(ROOT)}")